# Topic: SQL Injection Prevention - Prepared statements, parameterized database queries, and ORM safety mappings
 
## 1. THE SQL INJECTION VULNERABILITY
- **SQL Injection (SQLi)**: Occurs when user inputs are directly concatenated into a raw SQL query string.
  - If an input like `' OR '1'='1` is passed, it alters the mathematical logic of the database query, allowing attackers to bypass authentication or extract sensitive tables.
 
## 2. PARAMETERIZED QUERIES (PREPARED STATEMENTS)
- **The Solution**: Always use parameterized queries (placeholders like `?` or `%s`) provided by database drivers.
  - **How it works**: The database driver sends the query structure and parameter values separately to the database engine. The engine compiles the query template first, and guarantees that parameters are treated strictly as data literals, never as executable code instructions.
- **ORM Safety**: Object-Relational Mappers (like SQLAlchemy, Django ORM) use parameterized bindings under the hood for standard filter operations, protecting applications by default.
 
---


In [ ]:
import sqlite3

# 1. Setup in-memory sqlite lab database
conn = sqlite3.connect(":memory:")
cursor = conn.cursor()

# Create dummy users table
cursor.execute("CREATE TABLE users (id INTEGER PRIMARY KEY, username TEXT, password TEXT, is_admin INTEGER)")
cursor.execute("INSERT INTO users (username, password, is_admin) VALUES ('admin', 'SuperSecure123', 1)")
cursor.execute("INSERT INTO users (username, password, is_admin) VALUES ('alice', 'Pass321', 0)")
conn.commit()



In [ ]:
# 2. Vulnerable Login Query (String Concatenation)
def vulnerable_login(username, password):
    """Vulnerable implementation using raw string interpolation."""
    # DANGEROUS! Direct concatenation of variables
    query = f"SELECT * FROM users WHERE username = '{username}' AND password = '{password}'"
    print(f"[QUERY EXECUTED]: {query}")
    
    cursor.execute(query)
    result = cursor.fetchone()
    return result is not None

print("--- Vulnerable Auth Attempt (Normal User) ---")
print(f"Login success (alice): {vulnerable_login('alice', 'Pass321')}")

print("\n--- SQL Injection Attack Attempt ---")
# Attacker supplies: admin' --
# The '--' comments out the remaining password validation in SQL!
attacker_user = "admin' --"
print(f"Login success (SQLi): {vulnerable_login(attacker_user, 'invalid_password')}")
# Expected: True! Authentication bypassed without password!



In [ ]:
# 3. Secure Login Query (Parameterized prepared query)
def secure_login(username, password):
    """Secure implementation using parameterized query placeholders."""
    # The database driver handles character escaping and query structure isolation
    query = "SELECT * FROM users WHERE username = ? AND password = ?"
    print(f"[SECURE QUERY EXECUTED]: {query} | Parameters: ({username}, {password})")
    
    cursor.execute(query, (username, password))
    result = cursor.fetchone()
    return result is not None

print("\n--- Secure Query Validation ---")
print(f"Login success (alice): {secure_login('alice', 'Pass321')}")
print(f"Login success (SQLi):  {secure_login(attacker_user, 'invalid_password')}")
# Expected: False! Parameterization successfully blocked the attack.

# Close database
conn.close()
